Installations

In [34]:
import pandas as pd
import pytz
import re
import unicodedata
import nltk
import json
nltk.download('punkt')
nltk.download('punkt_tab')
from nltk.tokenize import word_tokenize
nltk.download('stopwords')
from nltk.corpus import stopwords
import numpy as np
import matplotlib.pyplot as plt
from scipy.cluster.hierarchy import linkage, fcluster
from sklearn.decomposition import PCA
from scipy.sparse import csr_matrix

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\fatim\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\fatim\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\fatim\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


commnets

In [ ]:
comments = pd.read_csv('instagram_comments.csv')
data = comments.loc[:,['post_id','created_at','username','text','parent_comment_id']]
data

Drop empty comments
Remove comments from pdamsuryasembada
Change date from epoch miliseconds
Remove taggings from comments

In [ ]:
data = data[data['username'].str.contains('pdamsuryasembada') == False]

data = data.drop_duplicates(subset=['text'])

local_timezone = pytz.timezone('Asia/Jakarta')
data['created_at'] = pd.to_datetime(data['created_at'], unit='s')
data['created_at'] = data['created_at'].dt.tz_localize('UTC').dt.tz_convert(local_timezone)
data['created_at'] = data['created_at'].dt.strftime('%Y-%m-%d %H:%M:%S %Z')

def remove_tags(text):
    if not isinstance(text, str):
        return ""
    return re.sub(r'@\w[\w\.]*', '', text).strip()

data['text'] = data['text'].apply(remove_tags)

data = data.dropna(subset=["text"])
data = data[data["text"].astype(str).str.strip() != ""]

cleaning

In [ ]:
PUNCT_TO_REMOVE = "!#$%&()*+,./:;<=>@[\\]^_{|}~`"
def remove_punctuation(text):
    """custom function to remove the punctuation"""
    return text.translate(str.maketrans('', '', PUNCT_TO_REMOVE))

data["text"] = data["text"].apply(lambda text: remove_punctuation(text))

def remove_emojis(text):
    """Removes standard unicode emojis from the text."""
    if not isinstance(text, str):
        return text

    emoji_pattern = re.compile(
        "["                     
        u"\U0001F600-\U0001F64F"
        u"\U0001F300-\U0001F5FF"
        u"\U0001F680-\U0001F6FF"
        u"\U0001F1E0-\U0001F1FF"
        u"\U0001F700-\U0001F77F"
        u"\U0001F780-\U0001F7FF"
        u"\U0001F800-\U0001F8FF"
        u"\U0001F900-\U0001F9FF"
        u"\U0001FA70-\U0001FAFF"
        u"\u2600-\u26FF"
        u"\u2700-\u27BF"
        u"\ufe0f"
        "]+",
        flags=re.UNICODE,
    )

    return emoji_pattern.sub(r'', text)

data['text'] = data['text'].apply(remove_emojis)
data['text'] = data['text'].str.strip()

def normalize_fancy_unicode(text):
    if not isinstance(text, str):
        return text

    normalized = unicodedata.normalize("NFKD", text)

    cleaned = "".join(
        ch for ch in normalized
        if not unicodedata.combining(ch)
    )
    return cleaned

data["text"] = data["text"].apply(normalize_fancy_unicode)

data = data.dropna(subset=["text"])
data = data[data["text"].astype(str).str.strip() != ""]

case folding

In [ ]:
data["text"] = data["text"].str.lower()

tokem

In [ ]:
data['text'] = data['text'].astype(str).apply(word_tokenize)

In [ ]:
def is_all_numbers(tokens):
    return all(re.fullmatch(r'\d+', tok) for tok in tokens)

data = data[~data['text'].apply(is_all_numbers)]


def normalize_mixed_number_token(token):
    if re.fullmatch(r'\d+', token):
        return "<num>"
    
    parts = re.findall(r'\d+|[a-zA-Z]+', token)

    if len(parts) == 1:
        return token

    parts = ["<num>" if p.isdigit() else p for p in parts]

    return " ".join(parts)


def normalize_tokens(token_list):
    normalized = []
    for tok in token_list:
        if tok.isdigit():
            normalized.append("<num>")
        elif re.search(r'\d', tok):
            normalized.append(normalize_mixed_number_token(tok))
        else:
            normalized.append(tok)
    return normalized

data["text"] = data["text"].apply(normalize_tokens)

In [ ]:
data_partial = data.copy()
data_partial.to_csv('all_tokenized.csv', index=False)
data_partial


formalize

In [ ]:
with open("dictionary/dict_template3_doneig.json", "r", encoding="utf-8") as f:
    formal_dict3 = json.load(f)

with open("dictionary/dict4.json", "r", encoding="utf-8") as f:
    formal_dict4 = json.load(f)

formal_dict = {**formal_dict3, **formal_dict4}

def apply_formalization(tokens, formal_dict):
    
    new_tokens = []
    for tok in tokens:
        if tok in formal_dict:
            replacement = formal_dict[tok]
            new_tokens.extend(replacement.split())
        else:
            new_tokens.append(tok)
    return new_tokens

data_partial["text"] = data_partial["text"].apply(lambda tokens: apply_formalization(tokens, formal_dict))

In [ ]:
data_partial

delete stopwor

In [ ]:
with open('stopwords_v3.txt', 'r', encoding='utf-8') as f:
    custom_stopwords = set([line.strip() for line in f if line.strip()])

def remove_stopwords(tokens, stopword_set):
    return [tok for tok in tokens if tok not in stopword_set and tok.strip() != ""]

data_partial["text"] = data_partial["text"].apply(lambda tokens: remove_stopwords(tokens, custom_stopwords))

stemmming

In [ ]:
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory
from Sastrawi.Stemmer.Stemmer import Stemmer
from Sastrawi.Stemmer.CachedStemmer import CachedStemmer
from Sastrawi.Stemmer.Cache.ArrayCache import ArrayCache
from Sastrawi.Dictionary.ArrayDictionary import ArrayDictionary

PROTECTED_WORDS = [
    'sememi',
]

factory = StemmerFactory()
base_words = factory.get_words_from_file()
combined_words = list(set(base_words + PROTECTED_WORDS))
custom_dict = ArrayDictionary(combined_words)
stemmer = CachedStemmer(ArrayCache(), Stemmer(custom_dict))

def stem_tokens(tokens):
    if not isinstance(tokens, list):
        return tokens
    return [stemmer.stem(tok) for tok in tokens]

data_partial["text"] = data_partial["text"].apply(stem_tokens)

In [ ]:
data_partial.to_csv("all_stemmed.csv", index=False)

In [35]:
comments = pd.read_csv('all_stemmed.csv')
data_partial = comments.loc[:,['post_id','created_at','username','text']]
data_partial = data_partial[(data_partial['created_at'] > '2025-10-01') & (data_partial['created_at'] < '2025-11-01')]
import ast

data_partial['text'] = data_partial['text'].apply(ast.literal_eval)

TF

In [36]:
from sklearn.feature_extraction.text import TfidfVectorizer
data_partial['processed_text'] = data_partial['text'].apply(
    lambda tokens: " ".join(tokens)
)
data_partial = data_partial[data_partial['processed_text'].str.strip() != ""]
vectorizer = TfidfVectorizer(
    tokenizer=str.split,
    ngram_range=(1,1),
    min_df=5,
    max_df=0.7,
    use_idf=False,
    norm=None
)

vectors = vectorizer.fit_transform(data_partial['processed_text'])
feature_names = vectorizer.get_feature_names_out()
vectors_dense = vectors.toarray()

d:\Kuliah\PA\pdam-scraper\scrape_instagram\venv\Lib\site-packages\sklearn\feature_extraction\text.py:517: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(


In [ ]:
print(f"Vector shape: {vectors.shape}")
print(f"nonzero: {vectors.nnz}")
row = vectors[0]
indices = row.indices
values = row.data

for i, v in zip(indices, values):
    print(feature_names[i], v) 

### TF Vector Normalization

#### Scenario 1
*Threshold*
1. find the greatest valued feature pof each comment
2. half the value and use it as treshold for that comment
3. for any feature with a value in that comment, if it is less than half of it, then set it as zero

In [37]:
vectors = vectorizer.fit_transform(data_partial['processed_text'])
feature_names = vectorizer.get_feature_names_out()
vectors_dense = vectors.toarray()

vectors_dense_old = vectors_dense.copy()
df_old = pd.DataFrame(vectors_dense_old, columns=feature_names)
df_old.insert(0, 'comment', data_partial['processed_text'].values)
df_old.to_csv("rec_tf_before_normalization.csv", index=False)
print(f"Old vector shape: {vectors_dense_old.shape}")

vectors_scene_1 = vectors_dense.copy()

for i in range(vectors_scene_1.shape[0]):
    row = vectors_scene_1[i]
    max_val = row.max()

    if max_val == 0:
        continue

    threshold = max_val / 2
    vectors_scene_1[i] = np.where(row >= threshold, row, 0.0)

non_zero_cols = np.any(vectors_scene_1 != 0, axis=0)
vectors_scene_1 = vectors_scene_1[:, non_zero_cols]
feature_names_scene_1 = feature_names[non_zero_cols]

print(f"New vector shape: {vectors_scene_1.shape}")
print(f"Features removed: {vectors_dense.shape[1] - vectors_scene_1.shape[1]}")

df_new = pd.DataFrame(vectors_scene_1, columns=feature_names_scene_1)
df_new.insert(0, 'comment', data_partial['processed_text'].values)
df_new.to_csv("rec_tf_after_normalization.csv", index=False)

Old vector shape: (1128, 241)
New vector shape: (1128, 241)
Features removed: 0


#### Scenario 2
*All ones*
1. for all feature with anon-zero value, ground them all to one
2. do this for all comments

In [38]:

vectors_scene_2 = (vectors_dense > 0).astype(int)
feature_names_scene_2 = feature_names
df_new_2 = pd.DataFrame(vectors_scene_2, columns=feature_names_scene_2)
df_new_2.insert(0, 'comment', data_partial['processed_text'].values)
df_new_2.to_csv("rec_tf_after_normalization_2.csv", index=False)

## Automatic Clustering

In [39]:
import matplotlib.pyplot as plt
from scipy.cluster.hierarchy import linkage, fcluster
from sklearn.metrics import silhouette_score
from sklearn.decomposition import PCA

VERSIONS = {
    "Before Normalization":              vectors_dense_old,
    "After Normalization (Threshold)":   vectors_scene_1,
    "After Normalization (Standardise 1)":    vectors_scene_2,
}

LINKAGE_METHODS = ['centroid']

v_results = {}

for method in LINKAGE_METHODS:
    for ver_name, vec in VERSIONS.items():
        run_key = f"{method.upper()} {ver_name}"

        print(f"\n{'='*60}")
        print(f"Linkage: {method}  |  Scenario: {ver_name}")
        print(f"Shape: {vec.shape}")
        print('-'*60)

        N = vec.shape[0] - 1
        Z = linkage(vec, method=method, metric='euclidean')
        k_list = list(range(N, 1, -1))
        grand_mean = vec.mean(axis=0)

        sil         = []
        vw_list     = []
        vb_list     = []
        v_ratio     = []
        valley_k    = []
        valley_diff = []

        for k in k_list:
            labels          = fcluster(Z, t=k, criterion='maxclust')
            unique_clusters = np.unique(labels)

            delta_within  = 0.0
            delta_between = 0.0

            for cid in unique_clusters:
                pts = vec[labels == cid]
                ni  = pts.shape[0]
                if ni <= 1:
                    continue
                centroid  = pts.mean(axis=0)
                delta_sq  = np.sum(np.linalg.norm(pts - centroid, axis=1) ** 2) / (ni - 1)
                delta_within  += (ni - 1) * delta_sq
                delta_between += ni * np.linalg.norm(centroid - grand_mean) ** 2

            vw2 = delta_within  / (N - k) if (N - k) > 0 else np.nan
            vb2 = delta_between / (k - 1) if (k - 1) > 0 else np.nan
            vw_list.append(vw2)
            vb_list.append(vb2)
            v_ratio.append((vw2 / vb2) * 100 if (vb2 and vb2 > 0) else np.nan)

            sc = (silhouette_score(vec, labels, metric='euclidean')
                  if len(unique_clusters) >= 2 else np.nan)
            sil.append(sc)

        vw_list = np.array(vw_list)
        vb_list = np.array(vb_list)
        v_ratio = np.array(v_ratio)
        sil     = np.array(sil)

        # V-Ratio
        first    = np.diff(v_ratio)
        second   = np.diff(first)
        idx_curv = np.nanargmax(np.abs(second))
        k_curv   = k_list[idx_curv + 2]

        # Valley-tracing
        for i in range(1, len(v_ratio) - 1):
            prev = v_ratio[i - 1]
            curr = v_ratio[i]
            nxt  = v_ratio[i + 1]
            if np.isnan(prev) or np.isnan(curr) or np.isnan(nxt):
                continue
            if prev >= curr and nxt > curr:
                valley_k.append(k_list[i])
                partial_diff = (prev + nxt) - (2 * curr)
                valley_diff.append(partial_diff)

        print("Valley-tracing candidate k values:", valley_k)

        # Accu
        max_diff = None
        max_k    = None
        if valley_k:
            paired   = sorted(zip(valley_diff, valley_k), reverse=True)
            max_diff = paired[0][0]
            max_k    = paired[0][1]

            if len(paired) >= 2:
                second_diff = paired[1][0]
                second_k    = paired[1][1]
                accuracy    = max_diff / second_diff if second_diff != 0 else float('inf')
                print(f"Best k by valley-tracing:  {max_k}")
                print(f"k Closest Value to max ∂:  {second_k}")
                print(f"Max ∂:                     {max_diff:.4f}")
                print(f"Second Max ∂:              {second_diff:.4f}")
                print(f"Acuuracy:              {accuracy:.4f}")
            else:
                print(f"Only one valley found — k: {max_k}, ∂: {max_diff:.4f}") #gk bisa uji akurasi

        v_results[run_key] = {
            'method'     : method,
            'scenario'   : ver_name,
            'Z'          : Z,
            'vec'        : vec,
            'k_list'     : k_list,
            'vw_list'    : vw_list,
            'vb_list'    : vb_list,
            'v_ratio'    : v_ratio,
            'sil'        : sil,
            'k_curv'     : k_curv,
            'valley_k'   : valley_k,
            'valley_diff': max_diff,
            'best_k'     : max_k,
        }


Linkage: centroid  |  Scenario: Before Normalization
Shape: (1128, 241)
------------------------------------------------------------
Valley-tracing candidate k values: [920, 827, 793, 788, 786, 784, 781, 777, 770, 766, 756, 752, 744, 738, 731, 727, 721, 714, 708, 688, 684, 680, 677, 672, 670, 667, 665, 662, 660, 600, 595, 592, 586, 578, 573, 554, 552, 550, 546, 539, 534, 525, 522, 472, 463, 459, 442, 438, 432, 421, 419, 417, 410, 406, 401, 399, 391, 389, 360, 345, 343, 333, 316, 312, 305, 277, 269, 263, 235, 217, 214, 211, 198, 188, 184, 179, 166, 163, 154, 145, 133, 103, 68, 24, 22, 18, 7, 5]
Best k by valley-tracing:  24
k Closest Value to max ∂:  5
Max ∂:                     2232.2788
Second Max ∂:              465.6532
Acuuracy:              4.7939

Linkage: centroid  |  Scenario: After Normalization (Threshold)
Shape: (1128, 241)
------------------------------------------------------------
Valley-tracing candidate k values: [913, 819, 785, 783, 777, 774, 771, 767, 758, 754, 747, 

In [ ]:
for ver_name, res in v_results.items():
    v_ratio = res['v_ratio']
    sil     = res['sil']
    k_list  = res['k_list']
    k_curv  = res['k_curv']
    Z       = res['Z']
    vec     = res['vec']
    valley_k= res['valley_k']
    best_k  = res['best_k']

    fig, axes = plt.subplots(1, 1, figsize=(12, 6))
    fig.suptitle(ver_name, fontsize=15, fontweight='bold')

    axes.plot(k_list, v_ratio, 'bo-', linewidth=2, markersize=3)
    axes.set_title("Valley tracing")
    if valley_k:
        for vk in valley_k:
            axes.axvline(vk, color='orange', linestyle=':', alpha=0.6)
        axes.axvline(best_k, color='red', linestyle='--', label=f"optimal k={best_k}")
    axes.legend()

    plt.tight_layout()
    plt.show()

In [40]:
feature_map = {
    "Before Normalization": feature_names,
    "After Normalization (Threshold)": feature_names_scene_1,
    "After Normalization (ground 1)": feature_names_scene_2,
}

def display_culster_details(tfidf_matrix, feature_names, labels, k, ver_name=""):
    print(f"\n{'-'*30}")
    print(f"Cluster Details — {ver_name}  (k={k})")
    
    cluster_info = {}
    
    for cluster_id in range(1, k+1):
        indices = np.where(labels == cluster_id)[0]
        
        if len(indices) == 0:
            continue
        
        cluster_data = tfidf_matrix[indices]
        if hasattr(cluster_data, 'toarray'):
            cluster_data = cluster_data.toarray()
        cluster_mean = cluster_data.mean(axis=0)
        threshold = cluster_mean.max() / 2
        above_threshold = np.where(cluster_mean > threshold)[0]
        sorted_idx = above_threshold[np.argsort(-cluster_mean[above_threshold])]
        keywords = [(feature_names[i], cluster_mean[i]) for i in sorted_idx]
        
        cluster_info[cluster_id] = {
            'members': data_partial['processed_text'].iloc[indices].tolist(),
            'count': len(indices),
            'keywords': keywords
        }
    
    for cluster_id, info in cluster_info.items():
        print(f"\nCluster {cluster_id}  ({info['count']} members)")
        print(f"  Members: {info['members']}")
        kw_str = ", ".join([f"{w}({s:.3f})" for w, s in info['keywords'][:10]])
        print(f"  Keywords: {kw_str if kw_str else '—'}")
    
    return cluster_info


for ver_name, res in v_results.items():
    k_used = res.get('best_k') or res['k_curv']
    k_used = int(k_used)
    labels_opt = fcluster(res['Z'], t=k_used, criterion='maxclust')
    scenario = res['scenario']
    features = feature_map.get(scenario, feature_names)

    display_culster_details(
        tfidf_matrix=res['vec'],
        feature_names=features,
        labels=labels_opt,
        k=k_used,
        ver_name=ver_name
    )


------------------------------
Cluster Details — CENTROID Before Normalization  (k=24)

Cluster 1  (3 members)
  Members: ['pbi malam nyala mati air mati ad pemberitahuan bayi jual umkm hambat wilayah surabaya barat pdam mati tagih air bayar mati pusat', 'aduh pdam gayungsari barat area resto kampung steak mati nyala air nyala mati nyala rumah mati bagi air bingung tampung air habis bantu pdam', 'aduh pdam gayungsari barat area resto kampung steak bingung tampung air rumah habis mati nyala air nyala mati nyala rumah mati bagi air bantu pdam']
  Keywords: mati(3.333), air(2.667), nyala(2.333)

Cluster 2  (1103 members)
  Members: ['paham harga pasang pdam', 'rame', 'pdam jalan gresik mati', 'tugas wilayah ploso timur kali cek air preman ketuk pagar ugal-ugalan kayu pentung preman pagar tetangga', 'kali kirim komplain langsung pecat tugas', 'pdam air hati alir baik', 'kebraon susah aktivitas', 'laku', 'syarat lengkap proses pasang', 'karah mati air pemberitahuan tayamum', 'aju tanggal s